# Run the Whole Pipeline

This is the heart of automation: instead of running five notebooks by hand every day, one
notebook runs them all in sequence. A scheduler (cron on Mac/Linux, Task Scheduler on
Windows) can then run this one notebook on a timetable, with no human involved.

Steps, in order:

1. combine the daily files
2. validate and write a data-quality report
3. clean and standardise
4. build the daily summary report
5. archive the processed input files

Each step runs in its own fresh Python kernel (the engine behind a notebook), exactly as
if you had opened it and chosen Run All. If a step fails, the pipeline stops there.

Run it with **Run > Run All Cells**. It reads and writes files relative to this folder,
so keep the notebook in `05-data-automation/`.

In [ ]:
import os
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

BASE = os.getcwd()

STEPS = [
    "01_combine_csvs.ipynb",
    "02_validate_readings.ipynb",
    "03_clean_and_standardize.ipynb",
    "04_daily_summary_report.ipynb",
    "05_archive_files.ipynb",
]

In [ ]:
def run_step(step):
    """Execute one step notebook in a fresh kernel and print what it printed."""
    nb = nbformat.read(os.path.join(BASE, step), as_version=4)
    client = NotebookClient(nb, timeout=600, kernel_name="python3",
                            resources={"metadata": {"path": BASE}})
    client.execute()
    for cell in nb.cells:
        for output in cell.get("outputs", []):
            if output.get("output_type") == "stream":
                print(output["text"], end="")

In [ ]:
def run():
    for step in STEPS:
        print("\n" + "=" * 70)
        print(f"STEP: {step}")
        print("=" * 70)
        try:
            run_step(step)
        except CellExecutionError as error:
            print(error)
            print(f"\nPipeline stopped: {step} raised an error.")
            return False
    print("\nPipeline finished. See data/reports/ for the outputs.")
    return True

## Run the pipeline

In [ ]:
ok = run()

## Running it on a schedule

A scheduler cannot click Run All, so it uses the `jupyter nbconvert` command, which runs
a notebook from the terminal. The `--output-dir` option saves a copy of the finished run
(with all its printed output) as a log, and leaves this notebook unchanged:

```
cd /full/path/to/05-data-automation
jupyter nbconvert --to notebook --execute --output-dir data/logs run_pipeline.ipynb
```